In [ ]:
# /// script
# dependencies = [
#   "geopandas",
#   "lonboard",
#   "pyarrow",
#   "ipywidgets",
# ]
# ///

import geopandas as gpd
import ipywidgets as widgets
from lonboard import Map, PathLayer, PolygonLayer
from lonboard.basemap import CartoStyle, MaplibreBasemap

# 1. Download Datasets
reefs_url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_reefs.geojson"
marine_url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_geography_marine_polys.geojson"
land_url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_land.geojson"

reefs_gdf = gpd.read_file(reefs_url)
marine_gdf = gpd.read_file(marine_url)
land_gdf = gpd.read_file(land_url)

# 2. Enrich Coral Reefs with Real Names via Spatial Join
named_marine = marine_gdf[["geometry", "name"]].dropna(subset=["name"])
reefs_named = gpd.sjoin_nearest(reefs_gdf, named_marine, how="left")
reefs_named["Reef System"] = reefs_named["name"].fillna("Pacific Marine Reef")

# 3. Add Educational / Beginner-Friendly Info Dictionary
REEF_FACTS = {
    "Great Barrier Reef": {
        "Region": "Coral Sea, Queensland, Australia",
        "Key Species": "Sea Turtles, Manta Rays, Clownfish, Dugongs",
        "Ecosystem Highlight": "World's largest coral reef system, visible from space.",
        "Primary Threat": "Coral bleaching due to rising sea temperatures."
    },
    "Coral Sea": {
        "Region": "South Pacific Ocean",
        "Key Species": "Grey Reef Sharks, Nautilus, Giant Clams",
        "Ecosystem Highlight": "High biodiversity zone connecting Australia & Melanesia.",
        "Primary Threat": "Ocean acidification and invasive species."
    },
    "South Pacific Ocean": {
        "Region": "Polynesia / Oceania",
        "Key Species": "Humpback Whales, Hard Corals, Reef Sharks",
        "Ecosystem Highlight": "Remote atolls with exceptionally clear waters.",
        "Primary Threat": "Overfishing and storm damage from cyclones."
    }
}

# Map facts into dataset with fallbacks
def get_fact(name, field, default):
    return REEF_FACTS.get(name, {}).get(field, default)

reefs_named["Location"] = reefs_named["Reef System"].apply(lambda x: get_fact(x, "Region", "Tropical Marine Region"))
reefs_named["Key Marine Life"] = reefs_named["Reef System"].apply(lambda x: get_fact(x, "Key Species", "Tropical Fish, Sea Turtles, Hard & Soft Corals"))
reefs_named["Ecosystem Highlights"] = reefs_named["Reef System"].apply(lambda x: get_fact(x, "Ecosystem Highlight", "Vital underwater habitat supporting global ocean biodiversity."))
reefs_named["Major Threat"] = reefs_named["Reef System"].apply(lambda x: get_fact(x, "Primary Threat", "Climate change and ocean temperature spikes."))

# Reset index to completely eliminate '__index_level_0__' from the click panel
reefs_named = reefs_named.reset_index(drop=True)

# Keep ONLY user-friendly columns for the click panel & hover card
reefs_named = reefs_named[[
    "geometry", 
    "Reef System", 
    "Location", 
    "Key Marine Life", 
    "Ecosystem Highlights", 
    "Major Threat"
]]

# 4. Context Layer (Dark Landmasses)
land_layer = PolygonLayer.from_geopandas(
    land_gdf,
    get_fill_color=[35, 39, 47, 255],
    get_line_color=[60, 65, 75, 255],
    get_line_width=500,
)

# 5. Foreground Coral Layer (Crisp Solid White Reefs)
coral_layer = PathLayer.from_geopandas(
    reefs_named,
    get_color=[255, 255, 255, 255],        # Crisp Solid White
    get_width=3,                           # Base line width
    width_min_pixels=1,
    opacity=0.9,
    pickable=True,                         # Enables click/hover inspection
    auto_highlight=True,                   # Highlighting feature under cursor
    highlight_color=[0, 255, 200, 255]     # Glowing Cyan highlight on selection
)

# 6. Construct Map
m = Map(
    layers=[land_layer, coral_layer],
    basemap=MaplibreBasemap(style=CartoStyle.DarkMatterNoLabels),
    view_state={
        "longitude": 147.0,   # Centered over Great Barrier Reef
        "latitude": -18.0,
        "zoom": 4.5,
    },
    picking_radius=12
)

# 7. Dynamic Info Card Below Dashboard
info_box = widgets.HTML(
    value="<div style='background-color: #1e222b; padding: 12px; border-radius: 6px; color: white;'>"
          "<b>🪸 Interactive Reef Guide:</b> Hover or click on any white coral reef line on the map to explore ecosystem details."
          "</div>"
)

def on_coral_select(change):
    idx = change["new"]
    if idx is not None and idx >= 0:
        feature = reefs_named.iloc[idx]
        info_box.value = f"""
        <div style="background-color: #1e222b; padding: 14px; border-radius: 6px; border-left: 5px solid #00FFC8; color: white;">
            <b style="color: #00FFC8; font-size: 18px;">🪸 {feature['Reef System']}</b><br/>
            <p style="margin: 6px 0 4px 0;"><b>📍 Location:</b> {feature['Location']}</p>
            <p style="margin: 4px 0;"><b>🐠 Key Marine Life:</b> {feature['Key Marine Life']}</p>
            <p style="margin: 4px 0;"><b>✨ Highlights:</b> {feature['Ecosystem Highlights']}</p>
            <p style="margin: 4px 0 0 0;"><b>⚠️ Major Threat:</b> {feature['Major Threat']}</p>
        </div>
        """

# Connect click/hover selection event to dynamic info box
coral_layer.observe(on_coral_select, names="selected_index")

# 8. Render Application
widgets.VBox([
    widgets.HTML("<h2>🪸 Global Coral Reef Inspector</h2>"),
    info_box,
    m
])